# Homework 2 — Pizza Hut Chatbot with SQLite Database Tools

Extend the Pizza Hut chatbot to use a **real SQLite database** instead of hardcoded menu text.
The agent uses LangChain tools to query the menu and place orders — demonstrating how LLMs interact with databases.

```
User: "I want 2 large Pepperoni pizzas"
  -> Agent calls get_menu()         — checks what's available
  -> Agent calls place_order(...)   — inserts into orders table
  -> Agent replies with confirmation + total price
```

In [ ]:
!pip install langchain langchain-ollama gradio --quiet

## Cell 2 — Set Up SQLite Database

In [ ]:
import sqlite3

def get_connection():
    return sqlite3.connect('pizzahut.db')

# Create tables
with get_connection() as conn:
    conn.execute('DROP TABLE IF EXISTS menu')
    conn.execute('DROP TABLE IF EXISTS orders')

    conn.execute('''
        CREATE TABLE menu (
            id          INTEGER PRIMARY KEY AUTOINCREMENT,
            name        TEXT NOT NULL,
            size        TEXT NOT NULL,
            price       REAL NOT NULL,
            description TEXT
        )
    ''')

    conn.execute('''
        CREATE TABLE orders (
            id          INTEGER PRIMARY KEY AUTOINCREMENT,
            item_name   TEXT NOT NULL,
            size        TEXT NOT NULL,
            quantity    INTEGER NOT NULL,
            price       REAL NOT NULL,
            status      TEXT DEFAULT 'pending',
            created_at  TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    ''')

    # Seed menu data
    menu_items = [
        ('Margherita',    'Small',  8.0,  'tomato sauce, mozzarella, basil'),
        ('Margherita',    'Medium', 12.0, 'tomato sauce, mozzarella, basil'),
        ('Margherita',    'Large',  16.0, 'tomato sauce, mozzarella, basil'),
        ('Pepperoni',     'Small',  10.0, 'tomato sauce, mozzarella, pepperoni'),
        ('Pepperoni',     'Medium', 14.0, 'tomato sauce, mozzarella, pepperoni'),
        ('Pepperoni',     'Large',  18.0, 'tomato sauce, mozzarella, pepperoni'),
        ('BBQ Chicken',   'Small',  11.0, 'BBQ sauce, chicken, onions, mozzarella'),
        ('BBQ Chicken',   'Medium', 15.0, 'BBQ sauce, chicken, onions, mozzarella'),
        ('BBQ Chicken',   'Large',  19.0, 'BBQ sauce, chicken, onions, mozzarella'),
        ('Veggie Supreme','Small',  10.0, 'tomato sauce, bell peppers, mushrooms, olives'),
        ('Veggie Supreme','Medium', 14.0, 'tomato sauce, bell peppers, mushrooms, olives'),
        ('Veggie Supreme','Large',  18.0, 'tomato sauce, bell peppers, mushrooms, olives'),
        ('Meat Lovers',   'Small',  12.0, 'pepperoni, sausage, bacon, beef'),
        ('Meat Lovers',   'Medium', 16.0, 'pepperoni, sausage, bacon, beef'),
        ('Meat Lovers',   'Large',  20.0, 'pepperoni, sausage, bacon, beef'),
        ('Garlic Bread',  'Regular', 4.0, 'toasted bread with garlic butter'),
        ('Chicken Wings', 'Regular', 8.0, '6 pieces with dipping sauce'),
        ('Caesar Salad',  'Regular', 6.0, 'romaine, croutons, parmesan, caesar dressing'),
        ('Coca-Cola',     'Regular', 2.0, 'chilled 330ml can'),
        ('Sprite',        'Regular', 2.0, 'chilled 330ml can'),
        ('Water',         'Regular', 1.0, '500ml bottle'),
    ]

    conn.executemany(
        'INSERT INTO menu (name, size, price, description) VALUES (?, ?, ?, ?)',
        menu_items
    )
    conn.commit()

# Verify
with get_connection() as conn:
    rows = conn.execute('SELECT * FROM menu').fetchall()
    print(f'Database ready — {len(rows)} menu items loaded.')
    print(f'{"ID":<4} {"Name":<15} {"Size":<8} {"Price":<7} Description')
    print('-' * 65)
    for row in rows:
        print(f'{row[0]:<4} {row[1]:<15} {row[2]:<8} ${row[3]:<6} {row[4]}')

## Cell 3 — Define Tools

Three tools the agent can use:
| Tool | What it does |
|------|--------------|
| `get_menu` | Reads all items from the `menu` table |
| `place_order` | Validates item and inserts a row into `orders` |
| `get_orders` | Shows all orders placed so far |

In [ ]:
from langchain_core.tools import tool

@tool
def get_menu() -> str:
    """Fetches all available items from the Pizza Hut menu with prices and descriptions."""
    with get_connection() as conn:
        rows = conn.execute('SELECT name, size, price, description FROM menu ORDER BY name, size').fetchall()

    if not rows:
        return 'Menu is currently unavailable.'

    lines = ['PIZZA HUT MENU', '=' * 50]
    current_name = None
    for name, size, price, description in rows:
        if name != current_name:
            lines.append(f'\n{name}')
            lines.append(f'  Ingredients: {description}')
            current_name = name
        lines.append(f'  {size:<10} ${price:.2f}')
    return '\n'.join(lines)


@tool
def place_order(item_name: str, size: str, quantity: int) -> str:
    """Places an order for a Pizza Hut menu item. Requires the exact item name, size (Small/Medium/Large/Regular), and quantity."""
    with get_connection() as conn:
        # Look up the item in the menu
        row = conn.execute(
            'SELECT price FROM menu WHERE LOWER(name) = LOWER(?) AND LOWER(size) = LOWER(?)',
            (item_name, size)
        ).fetchone()

        if not row:
            # Try to suggest close matches
            available = conn.execute(
                'SELECT name, size FROM menu WHERE LOWER(name) LIKE LOWER(?)',
                (f'%{item_name}%',)
            ).fetchall()
            if available:
                suggestions = ', '.join(f'{n} ({s})' for n, s in available)
                return f"Sorry, '{item_name}' in size '{size}' is not on the menu. Did you mean: {suggestions}?"
            return f"Sorry, '{item_name}' is not on the menu. Use get_menu to see available items."

        unit_price = row[0]
        total_price = unit_price * quantity

        # Insert the order
        conn.execute(
            'INSERT INTO orders (item_name, size, quantity, price, status) VALUES (?, ?, ?, ?, ?)',
            (item_name, size, quantity, total_price, 'confirmed')
        )
        conn.commit()

    return (
        f'Order confirmed!\n'
        f'  Item     : {quantity}x {size} {item_name}\n'
        f'  Price    : ${unit_price:.2f} each\n'
        f'  Total    : ${total_price:.2f}\n'
        f'  Status   : Confirmed\n'
        f'  Delivery : 30-45 minutes'
    )


@tool
def get_orders() -> str:
    """Retrieves all orders placed so far in this session, including item names, quantities, and prices."""
    with get_connection() as conn:
        rows = conn.execute(
            'SELECT item_name, size, quantity, price, status, created_at FROM orders ORDER BY id'
        ).fetchall()

    if not rows:
        return 'No orders have been placed yet.'

    lines = ['YOUR ORDERS', '=' * 50]
    grand_total = 0.0
    for i, (name, size, qty, price, status, created_at) in enumerate(rows, 1):
        lines.append(f'{i}. {qty}x {size} {name} — ${price:.2f} [{status}]')
        grand_total += price
    lines.append(f'\nGrand Total: ${grand_total:.2f}')
    return '\n'.join(lines)


# Quick direct tests
print('--- Testing get_menu ---')
print(get_menu.invoke({})[:300], '...')  # preview first 300 chars

print('\n--- Testing place_order ---')
print(place_order.invoke({'item_name': 'Pepperoni', 'size': 'Large', 'quantity': 2}))

print('\n--- Testing get_orders ---')
print(get_orders.invoke({}))

## Cell 4 — System Prompt

In [ ]:
SYSTEM_PROMPT = """You are a friendly and helpful ordering assistant for Pizza Hut.

You have access to these tools:
- get_menu: call this to show the customer what is available
- place_order: call this to place an order — requires item name, size, and quantity
- get_orders: call this to show the customer what they have ordered so far

Rules:
- Always use get_menu before recommending items — never guess prices
- Always confirm the item name, size, and quantity before calling place_order
- After placing an order, use get_orders to show the full order summary
- Be warm and helpful — suggest deals when relevant (Family Deal: 2 Large Pizzas + Garlic Bread + 2 Drinks = $35)
- If the customer asks about something unrelated to Pizza Hut, politely redirect
"""

print('System prompt set.')

## Cell 5 — LLM + Tool Setup

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model='llama3.2', temperature=0.7)

tools = [get_menu, place_order, get_orders]
tool_map = {t.name: t for t in tools}
llm_with_tools = llm.bind_tools(tools)

print('LLM bound with tools:', list(tool_map.keys()))

## Cell 6 — Agent Loop Function

The loop:
```
User message
  -> LLM decides to call a tool
  -> Tool executes against SQLite
  -> Result sent back to LLM
  -> LLM gives final natural language reply
```

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

def run_agent(user_message: str, history: list) -> str:
    """
    Run the agent loop for a single user turn.
    history is a list of dicts: [{'role': 'user'/'assistant', 'content': '...'}, ...]
    Returns the assistant's final reply as a string.
    """
    # Build message list from history
    messages = [SystemMessage(content=SYSTEM_PROMPT)]
    for msg in history:
        if msg['role'] == 'user':
            messages.append(HumanMessage(content=msg['content']))
        else:
            messages.append(AIMessage(content=msg['content']))
    messages.append(HumanMessage(content=user_message))

    # Agent loop
    while True:
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        # No tool calls — final answer
        if not response.tool_calls:
            return response.content

        # Execute each tool the model requested
        for tool_call in response.tool_calls:
            tool_fn = tool_map[tool_call['name']]
            tool_result = tool_fn.invoke(tool_call['args'])
            messages.append(
                ToolMessage(content=str(tool_result), tool_call_id=tool_call['id'])
            )

print('Agent loop ready.')

## Cell 7 — Test Without UI

Simulate a full customer conversation to verify everything works end-to-end.

In [ ]:
# Reset orders for a clean test
with get_connection() as conn:
    conn.execute('DELETE FROM orders')
    conn.commit()

history = []

queries = [
    "What's on the menu?",
    "I'd like to order 2 Large Pepperoni pizzas",
    "Also add a Garlic Bread",
    "Show me my orders",
    "How much would a Medium BBQ Chicken cost?",
]

for query in queries:
    print(f'Customer : {query}')
    reply = run_agent(query, history)
    history.append({'role': 'user',      'content': query})
    history.append({'role': 'assistant', 'content': reply})
    print(f'Bot      : {reply}')
    print()

## Cell 8 — Gradio UI

A chat interface identical in structure to the original Pizza Hut chatbot,
but now backed by real SQLite tools.

In [ ]:
import gradio as gr

def chat(user_message: str, history: list):
    reply = run_agent(user_message, history)
    history.append({'role': 'user',      'content': user_message})
    history.append({'role': 'assistant', 'content': reply})
    return '', history

with gr.Blocks(title='Pizza Hut Order Bot') as demo:
    gr.Markdown("""
    # Pizza Hut Order Bot
    Welcome! Ask me about the menu, place an order, or check your current orders.
    """)

    chatbot = gr.Chatbot(height=450, label='Pizza Hut Assistant', type='messages')
    msg = gr.Textbox(placeholder="What can I get you today?", label='You')
    clear = gr.Button('Clear Chat')

    msg.submit(chat, [msg, chatbot], [msg, chatbot])
    clear.click(lambda: [], outputs=chatbot)

demo.launch(theme=gr.themes.Soft())